# 📊 Dataset Comparison: SUN Database vs LDPolypVideo
### Deep Learning-Based Polyp Segmentation in Colonoscopy

This notebook trains the **same U-Net baseline** on both datasets and compares:
- **Training Loss** per epoch
- **Validation Accuracy**
- **Dice Coefficient**
- **IoU (Intersection over Union)**

## ⚙️ 0. Setup & Imports

In [ ]:

import os
import cv2
import glob
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import torch.optim as optim

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

# ---- Paths ----
PROJECT_ROOT = Path('/Users/alokkumarshukla/Desktop/Major Project 1')
SUN_PATH = PROJECT_ROOT / 'Sun' / 'sundatabase_positive_part1'
LD_TRAIN_PATH = PROJECT_ROOT / 'LD Polyp' / 'TrainValid'
LD_TEST_PATH = PROJECT_ROOT / 'LD Polyp' / 'Test'

# Full experiment settings for final dataset/model comparison.
# Reduce these values only if you need a quick CPU demo run.
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 8
EPOCHS = 5
SUBSET = 1000
BASE_CHANNELS = 64


## 📁 1. Dataset Loaders

In [ ]:

def read_rgb_image(img_path):
    """Read an image with OpenCV and return RGB format."""
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f'Could not read image: {img_path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def add_box_to_mask(mask, box):
    """Add one xmin,ymin,xmax,ymax bounding box to a binary mask."""
    h, w = mask.shape[:2]
    x1, y1, x2, y2 = [int(float(v)) for v in box[:4]]
    x_min, x_max = sorted((max(0, min(w, x1)), max(0, min(w, x2))))
    y_min, y_max = sorted((max(0, min(h, y1)), max(0, min(h, y2))))
    if x_max > x_min and y_max > y_min:
        mask[y_min:y_max, x_min:x_max] = 1


class BoundingBoxSegmentationDataset(Dataset):
    """Common image + bounding-box pseudo-mask dataset behavior."""
    def __init__(self, image_size=IMAGE_SIZE):
        self.image_size = image_size
        self.samples = []

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, boxes = self.samples[idx]
        img = read_rgb_image(img_path)
        h, w = img.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)

        for box in boxes:
            add_box_to_mask(mask, box)

        img = cv2.resize(img, self.image_size).astype(np.float32) / 255.0
        mask = cv2.resize(mask, self.image_size, interpolation=cv2.INTER_NEAREST)
        img = np.transpose(img, (2, 0, 1))

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32).unsqueeze(0),
        )


# ──────────────────────────────────────────────────
#  SUN Database Loader
#  Annotation format: "filename xmin,ymin,xmax,ymax,label"
# ──────────────────────────────────────────────────
class SUNPolypDataset(BoundingBoxSegmentationDataset):
    def __init__(self, root_dir, image_size=IMAGE_SIZE):
        super().__init__(image_size=image_size)
        root_dir = Path(root_dir)
        anno_dir = root_dir / 'annotation_txt'

        if not anno_dir.exists():
            raise FileNotFoundError(f'SUN annotation folder not found: {anno_dir}')

        for txt_file in sorted(anno_dir.glob('*.txt')):
            case_name = txt_file.stem
            case_dir = root_dir / case_name
            if not case_dir.exists():
                continue

            with txt_file.open('r') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split()
                    if len(parts) < 2:
                        continue

                    coords = parts[1].split(',')
                    if len(coords) < 4:
                        continue

                    img_path = case_dir / parts[0]
                    if img_path.exists():
                        self.samples.append((img_path, [[int(float(c)) for c in coords[:4]]]))


# ──────────────────────────────────────────────────
#  LDPolypVideo Loader
#  Annotation format per file:
#    Line 0 : number of polyps (0 = no polyp)
#    Lines 1+: "xmin ymin xmax ymax" per polyp
# ──────────────────────────────────────────────────
class LDPolypDataset(BoundingBoxSegmentationDataset):
    def __init__(self, root_dir, image_size=IMAGE_SIZE, skip_empty=True):
        super().__init__(image_size=image_size)
        root_dir = Path(root_dir)
        img_root = root_dir / 'Images'
        anno_root = root_dir / 'Annotations'

        if not img_root.exists():
            raise FileNotFoundError(f'LDPolyp image folder not found: {img_root}')
        if not anno_root.exists():
            raise FileNotFoundError(f'LDPolyp annotation folder not found: {anno_root}')

        for img_dir in sorted([p for p in img_root.iterdir() if p.is_dir()], key=lambda p: p.name):
            anno_dir = anno_root / img_dir.name
            if not anno_dir.exists():
                continue

            for img_file in sorted(img_dir.glob('*.jpg')):
                anno_file = anno_dir / f'{img_file.stem}.txt'
                if not anno_file.exists():
                    continue

                with anno_file.open('r') as f:
                    lines = [line.strip().replace('\r', '') for line in f if line.strip()]

                try:
                    n_polyps = int(lines[0]) if lines else 0
                except ValueError:
                    n_polyps = 0

                if skip_empty and n_polyps == 0:
                    continue

                boxes = []
                for line in lines[1:n_polyps + 1]:
                    coords = line.replace(',', ' ').split()
                    if len(coords) >= 4:
                        boxes.append([int(float(c)) for c in coords[:4]])

                if boxes or not skip_empty:
                    self.samples.append((img_file, boxes))


# Load datasets
sun_dataset = SUNPolypDataset(SUN_PATH)
ld_dataset = LDPolypDataset(LD_TRAIN_PATH)
ld_test_dataset = LDPolypDataset(LD_TEST_PATH)

print(f'SUN Dataset        : {len(sun_dataset):,} annotated frames')
print(f'LDPolyp TrainValid : {len(ld_dataset):,} annotated frames')
print(f'LDPolyp Test       : {len(ld_test_dataset):,} annotated frames')


## 👁️ 2. Visualization — One Sample from Each Dataset

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Dataset Sample Comparison', fontsize=16, fontweight='bold')

for row, (ds, name) in enumerate([(sun_dataset, 'SUN Database'), (ld_dataset, 'LDPolypVideo')]):
    if len(ds) == 0:
        axes[row, 0].set_title(f'{name} — No samples found')
        axes[row, 1].set_title(f'{name} — No mask available')
        axes[row, 0].axis('off')
        axes[row, 1].axis('off')
        continue

    img, mask = ds[0]
    axes[row, 0].imshow(img.permute(1, 2, 0).numpy())
    axes[row, 0].set_title(f'{name} — Frame')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(mask.squeeze().numpy(), cmap='hot')
    axes[row, 1].set_title(f'{name} — Pseudo-Mask (Bounding Box)')
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()


## 🧠 3. Segmentation Models & Metrics (Shared for Both Datasets)


In [ ]:

def calculate_metrics(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    target = target.float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    dice = (2. * intersection + 1e-6) / (union + 1e-6)
    iou = (intersection + 1e-6) / (union - intersection + 1e-6)
    acc = (pred == target).sum() / torch.numel(pred)
    return acc.item(), dice.item(), iou.item()


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class BasicUNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, base=BASE_CHANNELS):
        super().__init__()
        self.inc = DoubleConv(n_channels, base)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base, base * 2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base * 2, base * 4))
        self.up1 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.conv_up1 = DoubleConv(base * 4, base * 2)
        self.up2 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.conv_up2 = DoubleConv(base * 2, base)
        self.outc = nn.Conv2d(base, n_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x = self.conv_up1(torch.cat([x2, self.up1(x3)], dim=1))
        x = self.conv_up2(torch.cat([x1, self.up2(x)], dim=1))
        return self.outc(x)


class AttentionBlock(nn.Module):
    def __init__(self, gate_ch, skip_ch, inter_ch):
        super().__init__()
        self.gate = nn.Sequential(nn.Conv2d(gate_ch, inter_ch, 1), nn.BatchNorm2d(inter_ch))
        self.skip = nn.Sequential(nn.Conv2d(skip_ch, inter_ch, 1), nn.BatchNorm2d(inter_ch))
        self.psi = nn.Sequential(nn.ReLU(inplace=True), nn.Conv2d(inter_ch, 1, 1), nn.Sigmoid())

    def forward(self, gate, skip):
        attn = self.psi(self.gate(gate) + self.skip(skip))
        return skip * attn


class AttentionUNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, base=BASE_CHANNELS):
        super().__init__()
        self.inc = DoubleConv(n_channels, base)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base, base * 2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base * 2, base * 4))
        self.up1 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.att1 = AttentionBlock(base * 2, base * 2, base)
        self.conv_up1 = DoubleConv(base * 4, base * 2)
        self.up2 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.att2 = AttentionBlock(base, base, max(1, base // 2))
        self.conv_up2 = DoubleConv(base * 2, base)
        self.outc = nn.Conv2d(base, n_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        u1 = self.up1(x3)
        x2_att = self.att1(u1, x2)
        x = self.conv_up1(torch.cat([x2_att, u1], dim=1))
        u2 = self.up2(x)
        x1_att = self.att2(u2, x1)
        x = self.conv_up2(torch.cat([x1_att, u2], dim=1))
        return self.outc(x)


class UNetPlusPlus(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, base=BASE_CHANNELS):
        super().__init__()
        self.x00 = DoubleConv(n_channels, base)
        self.x10 = DoubleConv(base, base * 2)
        self.x20 = DoubleConv(base * 2, base * 4)
        self.x01 = DoubleConv(base + base, base)
        self.x11 = DoubleConv(base * 2 + base * 2, base * 2)
        self.x02 = DoubleConv(base + base + base, base)
        self.pool = nn.MaxPool2d(2)
        self.up10 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.up20 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.up11 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.outc = nn.Conv2d(base, n_classes, 1)

    def forward(self, x):
        x00 = self.x00(x)
        x10 = self.x10(self.pool(x00))
        x20 = self.x20(self.pool(x10))
        x01 = self.x01(torch.cat([x00, self.up10(x10)], dim=1))
        x11 = self.x11(torch.cat([x10, self.up20(x20)], dim=1))
        x02 = self.x02(torch.cat([x00, x01, self.up11(x11)], dim=1))
        return self.outc(x02)


class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)),
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=2, dilation=2), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)),
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=4, dilation=4), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)),
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=6, dilation=6), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)),
        ])
        self.project = nn.Sequential(nn.Conv2d(out_ch * 4, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.project(torch.cat([branch(x) for branch in self.branches], dim=1))


class DeepLabV3Plus(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, base=BASE_CHANNELS):
        super().__init__()
        self.enc1 = DoubleConv(n_channels, base)
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base, base * 2))
        self.enc3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(base * 2, base * 4))
        self.aspp = ASPP(base * 4, base * 2)
        self.low_project = nn.Sequential(nn.Conv2d(base, base, 1), nn.BatchNorm2d(base), nn.ReLU(inplace=True))
        self.decoder = nn.Sequential(DoubleConv(base * 3, base * 2), nn.Conv2d(base * 2, n_classes, 1))

    def forward(self, x):
        low = self.enc1(x)
        x = self.enc2(low)
        x = self.enc3(x)
        x = self.aspp(x)
        x = torch.nn.functional.interpolate(x, size=low.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([self.low_project(low), x], dim=1)
        return self.decoder(x)


MODEL_FACTORIES = {
    'Basic U-Net': BasicUNet,
    'Attention U-Net': AttentionUNet,
    'U-Net++': UNetPlusPlus,
    'DeepLabV3+': DeepLabV3Plus,
}

print(f'Defined {len(MODEL_FACTORIES)} segmentation models: {", ".join(MODEL_FACTORIES)}')


## 🏋️ 4. Training Function (Reusable)

In [ ]:

def train_and_evaluate(dataset, dataset_name, model_factory=BasicUNet, model_name='Basic U-Net', subset_size=SUBSET, epochs=EPOCHS):
    sample_count = min(subset_size, len(dataset))
    if sample_count < 2:
        raise ValueError(f'{dataset_name} needs at least 2 samples, found {sample_count}.')

    print(f'\n{"="*72}')
    print(f'  Training {model_name} on {dataset_name} ({sample_count} samples)')
    print(f'{"="*72}')

    indices = list(range(sample_count))
    sub = Subset(dataset, indices)
    train_n = max(1, int(0.8 * len(sub)))
    val_n = len(sub) - train_n
    if val_n == 0:
        train_n -= 1
        val_n = 1

    split_generator = torch.Generator().manual_seed(SEED)
    train_ds, val_ds = torch.utils.data.random_split(sub, [train_n, val_n], generator=split_generator)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    torch.manual_seed(SEED)
    model = model_factory().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    history = {'loss': [], 'acc': [], 'dice': [], 'iou': []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        v_acc = v_dice = v_iou = 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                a, d, i = calculate_metrics(model(imgs), masks)
                v_acc += a
                v_dice += d
                v_iou += i

        avg_loss = total_loss / len(train_loader)
        avg_acc = v_acc / len(val_loader)
        avg_dice = v_dice / len(val_loader)
        avg_iou = v_iou / len(val_loader)

        history['loss'].append(avg_loss)
        history['acc'].append(avg_acc)
        history['dice'].append(avg_dice)
        history['iou'].append(avg_iou)

        print(
            f'  Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | '
            f'Acc: {avg_acc:.4f} | Dice: {avg_dice:.4f} | IoU: {avg_iou:.4f}'
        )

    return history

print('Training function ready for multi-model comparison.')


## 🚀 5. Run Training Across All Models and Both Datasets


In [ ]:

datasets_to_compare = {
    'SUN Database': sun_dataset,
    'LDPolypVideo': ld_dataset,
}

all_histories = {}
for model_name, model_factory in MODEL_FACTORIES.items():
    all_histories[model_name] = {}
    for dataset_name, dataset in datasets_to_compare.items():
        all_histories[model_name][dataset_name] = train_and_evaluate(
            dataset,
            dataset_name,
            model_factory=model_factory,
            model_name=model_name,
        )

# Backward-compatible names for the original Basic U-Net charts.
sun_history = all_histories['Basic U-Net']['SUN Database']
ld_history = all_histories['Basic U-Net']['LDPolypVideo']


## 📈 6. Results Comparison — Charts


In [ ]:

epochs_range = range(1, EPOCHS + 1)
metrics = [
    ('loss', 'Training Loss', 'Loss'),
    ('acc', 'Validation Accuracy', 'Accuracy'),
    ('dice', 'Dice Coefficient', 'Dice Score'),
    ('iou', 'IoU Score', 'IoU'),
]

# Original Basic U-Net dataset comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SUN Database vs LDPolypVideo — Basic U-Net Baseline', fontsize=15, fontweight='bold')

for ax, (key, title, ylabel) in zip(axes.flatten(), metrics):
    ax.plot(epochs_range, sun_history[key], 'o-', color='#2196F3', linewidth=2, markersize=6, label='SUN Database')
    ax.plot(epochs_range, ld_history[key], 's-', color='#FF5722', linewidth=2, markersize=6, label='LDPolypVideo')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xticks(list(epochs_range))

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'comparison_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Basic U-Net chart saved as: {PROJECT_ROOT / "comparison_plot.png"}')

# Multi-model final metric comparison
model_names = list(MODEL_FACTORIES.keys())
dataset_names = list(datasets_to_compare.keys())
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Final Segmentation Scores by Model and Dataset', fontsize=15, fontweight='bold')

x = np.arange(len(model_names))
width = 0.36
for ax, metric_key, title, ylabel in [
    (axes[0], 'dice', 'Final Dice Score', 'Dice'),
    (axes[1], 'iou', 'Final IoU Score', 'IoU'),
]:
    for offset, dataset_name in zip([-width/2, width/2], dataset_names):
        scores = [all_histories[model][dataset_name][metric_key][-1] for model in model_names]
        ax.bar(x + offset, scores, width, label=dataset_name)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=20, ha='right')
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    ax.legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'multi_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Multi-model chart saved as: {PROJECT_ROOT / "multi_model_comparison.png"}')


## 📋 7. Final Results Summary Table


In [ ]:

summary_rows = []
for model_name in MODEL_FACTORIES:
    for dataset_name in datasets_to_compare:
        history = all_histories[model_name][dataset_name]
        summary_rows.append({
            'model': model_name,
            'dataset': dataset_name,
            'best_train_loss': min(history['loss']),
            'best_val_accuracy': max(history['acc']),
            'best_dice': max(history['dice']),
            'best_iou': max(history['iou']),
            'final_train_loss': history['loss'][-1],
            'final_accuracy': history['acc'][-1],
            'final_dice': history['dice'][-1],
            'final_iou': history['iou'][-1],
            'samples_used': min(SUBSET, len(datasets_to_compare[dataset_name])),
        })

print('=' * 118)
print(f'{"Model":<18} {"Dataset":<15} {"Best Dice":>10} {"Best IoU":>10} {"Final Dice":>12} {"Final IoU":>10} {"Final Acc":>10} {"Final Loss":>11}')
print('=' * 118)
for row in summary_rows:
    print(
        f'{row["model"]:<18} {row["dataset"]:<15} '
        f'{row["best_dice"]:>10.4f} {row["best_iou"]:>10.4f} '
        f'{row["final_dice"]:>12.4f} {row["final_iou"]:>10.4f} '
        f'{row["final_accuracy"]:>10.4f} {row["final_train_loss"]:>11.4f}'
    )
print('=' * 118)

csv_path = PROJECT_ROOT / 'model_comparison_results.csv'
with csv_path.open('w') as f:
    headers = list(summary_rows[0].keys())
    f.write(','.join(headers) + '\n')
    for row in summary_rows:
        f.write(','.join(str(row[h]) for h in headers) + '\n')

print(f'Results CSV saved as: {csv_path}')
print(f'Total SUN samples: {len(sun_dataset):,}')
print(f'Total LDPolypVideo TrainValid samples: {len(ld_dataset):,}')
print(f'Samples used per run: {min(SUBSET, len(sun_dataset)):,} SUN / {min(SUBSET, len(ld_dataset)):,} LDPolypVideo')
